In [ ]:
# Lambda Labs Optimized Custom YOLO Hyperparameter Tuning with Genetic Algorithm
import yaml
import os
import sys
import platform
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from ultralytics import YOLO, settings
from ultralytics.data.utils import check_det_dataset
import json
from datetime import datetime
from typing import Dict, Any, Optional, Tuple, List
import random
import pickle
import shutil
import time
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

# Lambda Labs specific: Set matplotlib backend
plt.switch_backend('Agg')
%matplotlib inline

# Display system info
print(f"🖥️ System Info:")
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
class LambdaGeneticYOLOTuner:
    """Lambda Labs optimized genetic algorithm YOLO tuner."""
    
    def __init__(self, config_path: str = "config.yaml"):
        self.config_path = Path(config_path)
        self.config = self._load_config()
        self.project_root = Path().resolve()
        self.base_tune_params = self.config['tune'].copy()
        self.base_tune_params.pop('search_space', None)
        self.base_tune_params.pop('fitness_weights', None)
        self.base_tune_params.pop('iterations', None)
        self.base_tune_params.pop('resume', None)
        
        # Genetic algorithm parameters - optimized for Lambda Labs
        self.population_size = 5  # Reduced for faster iterations
        self.elite_size = 2
        self.mutation_rate = 0.2
        self.crossover_rate = 0.8
        
        # Results tracking
        self.results_dir = None
        self.all_results = []
        self.generation_results = {}
        self.best_fitness = -float('inf')
        self.best_params = None
        
        # Lambda Labs specific setup
        self._setup_lambda_environment()
        self._verify_dataset()
    
    def _load_config(self) -> Dict[str, Any]:
        """Load configuration from YAML file."""
        try:
            with open(self.config_path, 'r') as f:
                config = yaml.safe_load(f)
            print(f"✅ Configuration loaded from: {self.config_path}")
            return config
        except Exception as e:
            print(f"❌ Error loading config: {e}")
            raise
    
    def _setup_lambda_environment(self):
        """Lambda Labs specific environment setup."""
        settings.update({"datasets_dir": str(self.project_root)})
        
        # Disable GUI components
        os.environ['DISPLAY'] = ''
        import matplotlib
        matplotlib.use('Agg')
        
        # Create results directory
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.results_dir = Path(self.config['output_config']['project']) / f"lambda_genetic_{timestamp}"
        self.results_dir.mkdir(parents=True, exist_ok=True)
        
        print(f"✅ Lambda Labs environment configured")
        print(f"📁 Results directory: {self.results_dir}")
    
    def _verify_dataset(self) -> None:
        """Verify dataset configuration."""
        dataset_yaml_path = Path(self.config["dataset_yaml_path"]).resolve()
        
        if not dataset_yaml_path.exists():
            raise FileNotFoundError(f"Dataset YAML not found: {dataset_yaml_path}")
        
        try:
            data = yaml.safe_load(dataset_yaml_path.read_text())
            modified = False
            
            if data.pop("path", None) is not None:
                modified = True
            
            if data.get("nc") != 1 or data.get("names") != ["powerline"]:
                data["nc"] = 1
                data["names"] = ["powerline"]
                modified = True
            
            if modified:
                dataset_yaml_path.write_text(yaml.safe_dump(data, sort_keys=False))
            
            check_det_dataset(str(dataset_yaml_path))
            print("✅ Dataset structure verified")
            
        except Exception as e:
            print(f"⚠️ Dataset verification warning: {e}")
    
    def _sample_parameter(self, param_name: str, param_range: List[float]) -> Any:
        """Sample a parameter value from its range."""
        min_val, max_val = param_range
        
        # Handle different parameter types
        if param_name in ['warmup_epochs', 'patience', 'save_period', 'close_mosaic', 'freeze']:
            return int(random.uniform(min_val, max_val))
        elif param_name in ['fliplr', 'flipud', 'mosaic', 'mixup', 'copy_paste']:
            return round(random.uniform(min_val, max_val), 3)
        else:
            return round(random.uniform(min_val, max_val), 6)
    
    def _generate_random_config(self) -> Dict[str, Any]:
        """Generate a random configuration from search space."""
        config = {}
        for param, param_range in self.tune_config['search_space'].items():
            config[param] = self._sample_parameter(param, param_range)
        return config
    
    def _crossover(self, parent1: Dict[str, Any], parent2: Dict[str, Any]) -> Dict[str, Any]:
        """Perform crossover between two parent configurations."""
        child = {}
        for param in parent1.keys():
            if random.random() < 0.5:
                child[param] = parent1[param]
            else:
                child[param] = parent2[param]
        return child
    
    def _mutate(self, config: Dict[str, Any]) -> Dict[str, Any]:
        """Mutate a configuration."""
        mutated = config.copy()
        for param, param_range in self.tune_config['search_space'].items():
            if random.random() < self.mutation_rate:
                mutated[param] = self._sample_parameter(param, param_range)
        return mutated
    
    def _calculate_fitness(self, metrics: Dict[str, float]) -> float:
        """Calculate fitness score based on metrics and weights."""
        weights = self.tune_config.get('fitness_weights', {
            'precision': 0.2,
            'recall': 0.6,
            'mAP50': 0.15,
            'mAP50_95': 0.05
        })
        
        fitness = 0.0
        for metric, weight in weights.items():
            if metric in metrics:
                fitness += metrics[metric] * weight
        
        return fitness
    
    def _extract_metrics(self, results) -> Dict[str, float]:
        """Extract metrics from training results."""
        if hasattr(results, 'results_dict'):
            metrics = {
                'precision': results.results_dict.get('metrics/precision(B)', 0),
                'recall': results.results_dict.get('metrics/recall(B)', 0),
                'mAP50': results.results_dict.get('metrics/mAP50(B)', 0),
                'mAP50_95': results.results_dict.get('metrics/mAP50-95(B)', 0),
            }
        else:
            # Fallback: try to read from CSV
            csv_path = Path(results.save_dir) / 'results.csv'
            if csv_path.exists():
                df = pd.read_csv(csv_path)
                if len(df) > 0:
                    last_row = df.iloc[-1]
                    metrics = {
                        'precision': last_row.get('metrics/precision(B)', 0),
                        'recall': last_row.get('metrics/recall(B)', 0),
                        'mAP50': last_row.get('metrics/mAP50(B)', 0),
                        'mAP50_95': last_row.get('metrics/mAP50-95(B)', 0),
                    }
                else:
                    metrics = {'precision': 0, 'recall': 0, 'mAP50': 0, 'mAP50_95': 0}
            else:
                metrics = {'precision': 0, 'recall': 0, 'mAP50': 0, 'mAP50_95': 0}
        return metrics

In [ ]:
def _train_single_config_lambda(self, config_id: int, hyperparams: Dict[str, Any], 
                               generation: int) -> Dict[str, Any]:
    """Train single config with Lambda Labs optimizations - CORRECTED VERSION."""
    print(f"🏃 Gen {generation}, Config {config_id}", end=' ')
    
    # CORRECTED: Start with base tune parameters (not hyperparams)
    train_params = self.base_tune_params.copy()
    
    # Add the search space parameters being tested
    train_params.update(hyperparams)
    
    # Add/override specific parameters for this run
    train_params.update({
        'data': str(Path(self.config["dataset_yaml_path"]).resolve()),
        'project': str(self.results_dir),
        'name': f"g{generation}_c{config_id}",
        'exist_ok': True,
        'verbose': False,
        'plots': False,  # Save memory
        'save': False,   # Save storage  
        'val': True,
        'patience': 10,  # Early stopping for efficiency
    })
    
    try:
        model = YOLO(self.config['model_type'])
        
        # Clear GPU cache
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        start_time = time.time()
        results = model.train(**train_params)
        train_time = time.time() - start_time
        
        # Extract metrics efficiently
        metrics = self._extract_metrics(results)
        fitness = self._calculate_fitness(metrics)
        
        print(f"✅ Fitness: {fitness:.4f} ({train_time:.1f}s)")
        
        # Clean up immediately
        if hasattr(results, 'save_dir') and Path(results.save_dir).exists():
            shutil.rmtree(results.save_dir, ignore_errors=True)
        
        # Clear GPU cache again
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        return {
            'config_id': config_id,
            'generation': generation,
            'hyperparams': hyperparams,
            'metrics': metrics,
            'fitness': fitness,
            'train_time': train_time,
            'timestamp': datetime.now().isoformat()
        }
        
    except Exception as e:
        print(f"❌ Error: {str(e)[:50]}")
        return {
            'config_id': config_id,
            'generation': generation,
            'hyperparams': hyperparams,
            'metrics': {'precision': 0, 'recall': 0, 'mAP50': 0, 'mAP50_95': 0},
            'fitness': 0,
            'train_time': 0,
            'error': str(e)
        }

def _evolve_population(self):
    """Evolve population using genetic operators."""
    new_population = []
    
    # Keep elite performers
    sorted_results = sorted(self.generation_results[len(self.generation_results) - 1], 
                          key=lambda x: x['fitness'], reverse=True)
    elite = [r['hyperparams'] for r in sorted_results[:self.elite_size]]
    new_population.extend(elite)
    
    # Generate rest through crossover and mutation
    while len(new_population) < self.population_size:
        if random.random() < self.crossover_rate and len(sorted_results) >= 2:
            # Crossover
            parent1 = random.choice(sorted_results[:self.population_size//2])['hyperparams']
            parent2 = random.choice(sorted_results[:self.population_size//2])['hyperparams']
            child = self._crossover(parent1, parent2)
            if random.random() < self.mutation_rate:
                child = self._mutate(child)
            new_population.append(child)
        else:
            # Random exploration
            new_population.append(self._generate_random_config())
    
    return new_population[:self.population_size]

def _save_results(self):
    """Save intermediate results to CSV."""
    if self.all_results:
        # Flatten results for CSV
        flattened_results = []
        for result in self.all_results:
            row = {
                'config_id': result['config_id'],
                'generation': result['generation'],
                'fitness': result['fitness'],
                'precision': result['metrics']['precision'],
                'recall': result['metrics']['recall'],
                'mAP50': result['metrics']['mAP50'],
                'mAP50_95': result['metrics']['mAP50_95'],
                'train_time': result['train_time'],
                'timestamp': result['timestamp']
            }
            # Add hyperparameters
            for param, value in result['hyperparams'].items():
                row[param] = value
            flattened_results.append(row)
        
        df = pd.DataFrame(flattened_results)
        df.to_csv(self.results_dir / 'tune_results.csv', index=False)

def _save_checkpoint(self, generation: int):
    """Save checkpoint for resume capability."""
    checkpoint = {
        'generation': generation,
        'all_results': self.all_results,
        'generation_results': self.generation_results,
        'best_fitness': self.best_fitness,
        'best_params': self.best_params,
        'random_state': random.getstate(),
        'numpy_state': np.random.get_state()
    }
    
    checkpoint_path = self.results_dir / 'checkpoint.pkl'
    with open(checkpoint_path, 'wb') as f:
        pickle.dump(checkpoint, f)

def _save_final_results(self):
    """Save final results and best hyperparameters."""
    if self.best_params:
        best_config = {
            '# Best hyperparameters from genetic algorithm tuning': None,
            '# Fitness score': float(self.best_fitness),
            '# Timestamp': datetime.now().isoformat(),
            '# Focused on': 'Maximizing recall for powerline detection',
            **self.best_params
        }
        
        with open(self.results_dir / 'best_hyperparameters.yaml', 'w') as f:
            yaml.dump(best_config, f, default_flow_style=False, sort_keys=False)

# Attach methods to class
LambdaGeneticYOLOTuner._train_single_config_lambda = _train_single_config_lambda
LambdaGeneticYOLOTuner._evolve_population = _evolve_population
LambdaGeneticYOLOTuner._save_results = _save_results
LambdaGeneticYOLOTuner._save_checkpoint = _save_checkpoint
LambdaGeneticYOLOTuner._save_final_results = _save_final_results

In [ ]:
def run_lambda_genetic_tuning(self) -> pd.DataFrame:
    """Run genetic tuning optimized for Lambda Labs."""
    total_iterations = self.tune_config['iterations']
    num_generations = max(1, total_iterations // self.population_size)
    
    print(f"🧬 Lambda Labs Genetic Algorithm Tuning")
    print(f"📊 Total iterations: {total_iterations}")
    print(f"👥 Population size: {self.population_size}")
    print(f"🔄 Generations: {num_generations}")
    
    # Time estimation
    epochs_per_config = self.tune_config['epochs']
    seconds_per_epoch = 15
    estimated_hours = (total_iterations * epochs_per_config * seconds_per_epoch) / 3600
    print(f"\n⏱️ Time estimate: {estimated_hours:.1f} hours")
    print(f"💰 Cost estimate: ~${estimated_hours * 1.29:.2f} (A100 40GB)")
    
    # Confirm
    if input("\n🤔 Continue? (y/N): ").lower() != 'y':
        print("❌ Cancelled")
        return pd.DataFrame()
    
    start_time = datetime.now()
    print(f"\n🕐 Started: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
    
    # Main loop
    for generation in range(num_generations):
        print(f"\n{'='*50}")
        print(f"🧬 GENERATION {generation + 1}/{num_generations}")
        
        # Generate population
        if generation == 0:
            population = [self._generate_random_config() for _ in range(self.population_size)]
        else:
            # Genetic operators
            population = self._evolve_population()
        
        # Train population
        generation_results = []
        for i, config in enumerate(population):
            result = self._train_single_config_lambda(i, config, generation)
            generation_results.append(result)
            self.all_results.append(result)
            
            # Update best
            if result['fitness'] > self.best_fitness:
                self.best_fitness = result['fitness']
                self.best_params = result['hyperparams']
                print(f"🏆 New best: {self.best_fitness:.4f}")
            
            # Save incrementally
            self._save_results()
            
            # Clear output periodically to prevent notebook bloat
            if i % 3 == 0:
                clear_output(wait=True)
                print(f"Generation {generation + 1}/{num_generations}, Config {i + 1}/{self.population_size}")
        
        self.generation_results[generation] = generation_results
        
        # Generation summary
        gen_fitness = [r['fitness'] for r in generation_results]
        print(f"\n📊 Gen {generation + 1}: Best={max(gen_fitness):.4f}, Mean={np.mean(gen_fitness):.4f}")
        
        # Save checkpoint
        self._save_checkpoint(generation)
    
    # Final summary
    end_time = datetime.now()
    print(f"\n{'='*50}")
    print(f"✅ COMPLETED")
    print(f"🏆 Best fitness: {self.best_fitness:.4f}")
    print(f"⏱️ Duration: {end_time - start_time}")
    print(f"📁 Results: {self.results_dir}")
    
    self._save_final_results()
    return pd.DataFrame(self.all_results)

# Attach method
LambdaGeneticYOLOTuner.run_lambda_genetic_tuning = run_lambda_genetic_tuning

In [ ]:
def analyze_lambda_results(self, results_df: pd.DataFrame):
    """Analyze results with Lambda Labs optimizations."""
    print(f"📊 Analyzing {len(results_df)} results...")
    
    # Create compact visualization
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # 1. Fitness progression
    axes[0, 0].plot(results_df.index, results_df['fitness'], alpha=0.7)
    axes[0, 0].axhline(y=results_df['fitness'].max(), color='red', linestyle='--', label='Best')
    axes[0, 0].set_title('Fitness Progression')
    axes[0, 0].set_xlabel('Configuration')
    axes[0, 0].set_ylabel('Fitness')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Generation performance
    if 'generation' in results_df.columns:
        gen_stats = results_df.groupby('generation')['fitness'].agg(['mean', 'max'])
        axes[0, 1].plot(gen_stats.index, gen_stats['mean'], label='Mean', marker='o')
        axes[0, 1].plot(gen_stats.index, gen_stats['max'], label='Max', marker='s')
        axes[0, 1].set_title('Fitness by Generation')
        axes[0, 1].set_xlabel('Generation')
        axes[0, 1].set_ylabel('Fitness')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Recall vs Precision
    scatter = axes[1, 0].scatter(results_df['precision'], results_df['recall'], 
                                c=results_df['fitness'], cmap='viridis', alpha=0.6)
    axes[1, 0].set_title('Recall vs Precision')
    axes[1, 0].set_xlabel('Precision')
    axes[1, 0].set_ylabel('Recall')
    axes[1, 0].grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=axes[1, 0], label='Fitness')
    
    # 4. Top 5 configs
    top_5 = results_df.nlargest(5, 'fitness')
    axes[1, 1].bar(range(len(top_5)), top_5['fitness'])
    axes[1, 1].set_title('Top 5 Configurations')
    axes[1, 1].set_xlabel('Rank')
    axes[1, 1].set_ylabel('Fitness')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(self.results_dir / 'lambda_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Print best config
    best_idx = results_df['fitness'].idxmax()
    best = results_df.iloc[best_idx]
    print(f"\n🏆 Best Configuration:")
    print(f"Fitness: {best['fitness']:.4f}")
    print(f"Recall: {best['recall']:.4f}")
    print(f"Precision: {best['precision']:.4f}")
    print(f"mAP50: {best['mAP50']:.4f}")

# Attach method
LambdaGeneticYOLOTuner.analyze_lambda_results = analyze_lambda_results

In [ ]:
# Initialize and run
tuner = LambdaGeneticYOLOTuner('config.yaml')

# Run genetic tuning
results_df = tuner.run_lambda_genetic_tuning()

# Analyze if successful
if not results_df.empty:
    tuner.analyze_lambda_results(results_df)
    
    # Display best hyperparameters
    if tuner.results_dir:
        best_params_file = tuner.results_dir / 'best_hyperparameters.yaml'
        if best_params_file.exists():
            print("\n🏆 BEST HYPERPARAMETERS:")
            print("="*50)
            with open(best_params_file, 'r') as f:
                print(f.read())
else:
    print("❌ No results to analyze")

In [ ]:
# Create download links for Lambda Labs
import zipfile
from IPython.display import FileLink

if tuner.results_dir and tuner.results_dir.exists():
    # Create zip file
    zip_path = f"genetic_tuning_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"
    
    with zipfile.ZipFile(zip_path, 'w') as zipf:
        for file in tuner.results_dir.rglob('*'):
            if file.is_file():
                zipf.write(file, file.relative_to(tuner.results_dir.parent))
    
    print(f"📦 Results packaged: {zip_path}")
    display(FileLink(zip_path))
    print("\n⚠️ Download results before terminating Lambda Labs instance!")